# 224 px GPU benchmark (Colab / Kaggle)

Set the runtime to **GPU** (T4/P100) before running. Push the updated `main` branch first, or replace `REPO_URL` with an accessible copy of this repository.

The dataset must contain three folders under one parent: `ResearchImage/`, `Images_v1/`, `Images_v2/`. On Colab put them in Google Drive; on Kaggle attach them as a dataset. The notebook links that parent to `data/dataset` and never copies images into the repo.

In [ ]:
import os
import sys
import shutil
import pathlib
import subprocess

COLAB = "google.colab" in sys.modules
KAGGLE = os.path.exists("/kaggle/input")
print("environment:", "colab" if COLAB else "kaggle" if KAGGLE else "local")

In [ ]:
REPO_URL = "https://github.com/HasanImamB/SkinDiseaseBD.git"
WORKDIR = "/content/SkinDiseaseBD" if COLAB else "/kaggle/working/SkinDiseaseBD"
repo = pathlib.Path(WORKDIR)
if repo.exists():
    subprocess.run(["git", "-C", WORKDIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)
print(repo)

In [ ]:
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

SEARCH_ROOTS = (
    ["/content/drive/MyDrive", "/content"]
    if COLAB
    else ["/kaggle/input"]
)
print("search roots:", SEARCH_ROOTS)

In [ ]:
REQUIRED = {"ResearchImage", "Images_v1", "Images_v2"}


def find_dataset(roots):
    for root in roots:
        base = pathlib.Path(root)
        if not base.exists():
            continue
        for candidate in base.rglob("ResearchImage"):
            parent = candidate.parent
            if all((parent / name).is_dir() for name in REQUIRED):
                return parent
    raise FileNotFoundError(f"no dataset with {sorted(REQUIRED)} under {roots}")


DATASET_DIR = find_dataset(SEARCH_ROOTS)
print("dataset:", DATASET_DIR)

In [ ]:
target = repo / "data" / "dataset"
target.parent.mkdir(parents=True, exist_ok=True)
if target.exists() or target.is_symlink():
    if not target.is_symlink():
        raise SystemExit(f"{target} exists and is not a symlink; remove it first")
    target.unlink()
target.symlink_to(DATASET_DIR, target_is_directory=True)
print("linked:", target, "->", DATASET_DIR)

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements_cloud.txt")],
    check=True,
)
import torch

print("torch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected; switch the runtime to GPU before training")
print("device:", torch.cuda.get_device_name(0))

In [ ]:
MODEL = "resnet50.a1_in1k"
RUN_NAME = "resnet50_224_balanced"
FOLDS = "0,1,2,3,4"
REPEATS = "0,1,2"
SEEDS = "42"
EPOCHS = 12
BATCH_SIZE = 32
PATIENCE = 3
print(MODEL, RUN_NAME, FOLDS, REPEATS, SEEDS, EPOCHS, BATCH_SIZE)

In [ ]:
cmd = [
    sys.executable,
    str(repo / "scripts" / "train.py"),
    "--model", MODEL,
    "--run-name", RUN_NAME,
    "--folds", FOLDS,
    "--repeats", REPEATS,
    "--seeds", SEEDS,
    "--epochs", str(EPOCHS),
    "--image-size", "224",
    "--batch-size", str(BATCH_SIZE),
    "--patience", str(PATIENCE),
    "--class-balanced",
    "--amp",
    "--resume",
    "--device", "cuda",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=repo, check=True)

In [ ]:
subprocess.run(
    [sys.executable, str(repo / "scripts" / "summarize_finetuned.py"), "--run-name", RUN_NAME],
    cwd=repo,
    check=True,
)

Outputs are written under `results/` in the repo. The last cell copies the summary, confusion matrix, OOF file and per-fold predictions to persistent storage (`/content/drive/MyDrive/SkinDiseaseBD_runs` on Colab, `/kaggle/working/run_output` on Kaggle).

In [ ]:
out_dir = pathlib.Path(
    "/content/drive/MyDrive/SkinDiseaseBD_runs"
    if COLAB
    else "/kaggle/working/run_output"
)
out_dir.mkdir(parents=True, exist_ok=True)
patterns = (
    f"finetuned/{RUN_NAME}_summary.json",
    f"finetuned/{RUN_NAME}_confusion_matrix.csv",
    f"predictions/finetuned_{RUN_NAME}_oof.csv",
    f"predictions/finetuned_{RUN_NAME}_fold*_test.csv",
)
copied = 0
for pattern in patterns:
    for path in (repo / "results").glob(pattern):
        shutil.copy2(path, out_dir / path.name)
        copied += 1
print(f"copied {copied} files to {out_dir}")